# 🗺️ Rumsey Map OCR — Detection Model Fine-tuning
Run each cell **top to bottom**. Training takes **4-6 hours** on a T4 GPU.

> **Before running:** `Runtime → Change runtime type → T4 GPU`

### What you need on Google Drive first:
```
My Drive/
└── Rumsey_OCR/
    ├── train_data/
    │   └── det/            ← upload this from your laptop
    │       ├── train_list.txt
    │       ├── val_list.txt
    │       └── train_images/
    └── det_icdar_finetune.yml   ← upload this config file
```


## Step 1 — Verify GPU


In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout or '❌ No GPU — go to Runtime → Change runtime type → T4 GPU')


## Step 2 — Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/Rumsey_OCR'

# Verify required files are on Drive
checks = {
    'Training data':  os.path.join(DRIVE_ROOT, 'train_data/det/train_list.txt'),
    'Val data':       os.path.join(DRIVE_ROOT, 'train_data/det/val_list.txt'),
    'Config file':    os.path.join(DRIVE_ROOT, 'det_icdar_finetune.yml'),
}
all_ok = True
for name, path in checks.items():
    exists = os.path.exists(path)
    status = '✅' if exists else '❌  MISSING'
    print(f'  {status}  {name}: {path}')
    if not exists: all_ok = False
if not all_ok:
    print('\n⚠️  Upload missing files to Drive before continuing!')
else:
    print('\n✅ All required files found!')


## Step 3 — Install PaddlePaddle GPU
*(Takes ~2 min — run once per session)*


In [ ]:
!pip install paddlepaddle-gpu==2.6.1.post120 \
    -f https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html -q

import paddle
print(f'PaddlePaddle: {paddle.__version__}')
print(f'GPU available: {paddle.is_compiled_with_cuda()}')


## Step 4 — Get PaddleOCR Toolkit
Clones the official PaddleOCR training tools. *(No need to upload — pulled from PaddlePaddle directly)*


In [ ]:
import os

PADDLEOCR_DIR = '/content/PaddleOCR'

if not os.path.exists(PADDLEOCR_DIR):
    !git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git {PADDLEOCR_DIR} -q
    print('Cloned PaddleOCR')
else:
    print('PaddleOCR already present')

# Install requirements
!pip install -r {PADDLEOCR_DIR}/requirements.txt -q
print('Requirements installed!')


## Step 5 — Setup Workspace


In [ ]:
import os, shutil

DRIVE_ROOT   = '/content/drive/MyDrive/Rumsey_OCR'
PADDLEOCR_DIR = '/content/PaddleOCR'
WORK_DIR      = '/content/workspace'
os.makedirs(WORK_DIR, exist_ok=True)

# Link training data from Drive into workspace
DET_DATA_SRC = os.path.join(DRIVE_ROOT, 'train_data/det')
DET_DATA_DST = os.path.join(WORK_DIR, 'train_data/det')
os.makedirs(os.path.dirname(DET_DATA_DST), exist_ok=True)
if not os.path.exists(DET_DATA_DST):
    os.symlink(DET_DATA_SRC, DET_DATA_DST)
    print(f'Linked train_data/det from Drive')

# Copy config into workspace (and fix paths to be absolute)
CONFIG_SRC = os.path.join(DRIVE_ROOT, 'det_icdar_finetune.yml')
CONFIG_DST = os.path.join(WORK_DIR, 'det_icdar_finetune.yml')
shutil.copy2(CONFIG_SRC, CONFIG_DST)

# Link output directory to Drive for REAL-TIME checkpoint saving
# This ensures that if Colab disconnects, your progress is saved on Drive!
DRIVE_OUT_DIR = os.path.join(DRIVE_ROOT, 'output/det_finetune')
LOCAL_OUT_DIR = os.path.join(WORK_DIR, 'output/det_finetune')
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(LOCAL_OUT_DIR), exist_ok=True)

if not os.path.exists(LOCAL_OUT_DIR):
    os.symlink(DRIVE_OUT_DIR, LOCAL_OUT_DIR)
    print(f'✅ Output directory symlinked to Drive: {DRIVE_OUT_DIR}')

# Patch paths in config to be absolute
with open(CONFIG_DST) as f:
    cfg = f.read()
cfg = cfg.replace('../train_data/det', DET_DATA_DST)
cfg = cfg.replace('../output/det_finetune', LOCAL_OUT_DIR)
with open(CONFIG_DST, 'w') as f:
    f.write(cfg)
print('Config ready at:', CONFIG_DST)
print('Workspace ready!')


## Step 6 — Download Pretrained Detection Model


In [ ]:
import os
WORK_DIR    = '/content/workspace'
PRETRAINED  = os.path.join(WORK_DIR, 'pretrained/ch_PP-OCRv4_det_train')
TAR_FILE    = os.path.join(WORK_DIR, 'pretrained/ch_PP-OCRv4_det_train.tar')

if not os.path.exists(PRETRAINED):
    os.makedirs(os.path.join(WORK_DIR, 'pretrained'), exist_ok=True)
    print('Downloading pretrained model...')
    URL = 'https://paddleocr.bj.bcebos.com/PP-OCRv4/chinese/ch_PP-OCRv4_det_train.tar'
    # Remove existing partial downloads
    if os.path.exists(TAR_FILE): os.remove(TAR_FILE)
    
    # Use curl (more reliable than wget in Colab)
    import subprocess
    subprocess.run(['curl', '-L', '--progress-bar', '-o', TAR_FILE, URL], check=False)
    
    size = os.path.getsize(TAR_FILE) if os.path.exists(TAR_FILE) else 0
    print(f'Downloaded: {size/1024/1024:.1f} MB')
    if size > 10_000_000:
        print('Verifying archive...')
        try:
            # Check if it's a valid tar file
            is_valid = subprocess.run(['tar', '-tf', TAR_FILE], capture_output=True).returncode == 0
            if is_valid:
                print('Extracting...')
                !tar -xf {TAR_FILE} -C {os.path.join(WORK_DIR, 'pretrained')}
                print('✅ Extracted successfully!')
            else:
                print('❌ Downloaded file is not a valid archive. Retrying download might be needed.')
        except:
            !tar -xf {TAR_FILE} -C {os.path.join(WORK_DIR, 'pretrained')}
    else:
        print('❌ Download failed or file too small.')
else:
    print('✅ Pretrained model already present')

# Patch config with pretrained model path
CONFIG_DST = os.path.join(WORK_DIR, 'det_icdar_finetune.yml')
if os.path.exists(CONFIG_DST):
    with open(CONFIG_DST) as f:
        cfg = f.read()
    cfg = cfg.replace('../pretrained/ch_PP-OCRv4_det_train/best_accuracy',
                      os.path.join(PRETRAINED, 'best_accuracy'))
    with open(CONFIG_DST, 'w') as f:
        f.write(cfg)
    print('Config updated with pretrained model path')
else:
    print('⚠️  Config file not found in workspace yet!')


## Step 7 — Verify Training Data


In [ ]:
WORK_DIR = '/content/workspace'
import os

for fname in ['train_list.txt', 'val_list.txt']:
    path = os.path.join(WORK_DIR, 'train_data/det', fname)
    if os.path.exists(path):
        with open(path) as f: n = sum(1 for _ in f)
        print(f'✅ {fname}: {n:,} images')
    else:
        print(f'❌ MISSING: {path}')


## Step 8 — Train Detection Model 🔥
This is the main cell. Takes **4-6 hours**.
> The session stays alive as long as this cell is running.

> **Tip:** Keep your browser tab open (or use Colab Pro for longer sessions).


In [ ]:
import os, yaml, glob, shutil

WORK_DIR      = '/content/workspace'
PADDLEOCR_DIR = '/content/PaddleOCR'
CONFIG        = os.path.join(WORK_DIR, 'det_icdar_finetune.yml')
CKPT_DIR      = os.path.join(WORK_DIR, 'output/det_finetune')

os.chdir(PADDLEOCR_DIR)

# ── 0. Wipe any checkpoint saved from a broken (NaN-loss) run ──
# Any checkpoint in det_finetune/ that predates this fix is worthless:
# gradients never flowed so weights were never updated, but the optimizer
# state was updated with zero/NaN gradients. Resuming from it would
# carry over that corrupted state. Wipe it now and always start fresh
# from the official pretrained PP-OCRv4 weights.
_bad_ckpts = glob.glob(os.path.join(CKPT_DIR, '*.pdparams')) + \
             glob.glob(os.path.join(CKPT_DIR, '*.pdopt'))
if _bad_ckpts:
    for _f in _bad_ckpts:
        os.remove(_f)
    print(f'🗑️  Removed {len(_bad_ckpts)} stale checkpoints (NaN-run artifacts) — starting fresh')
else:
    print('✓ No stale checkpoints found')

# ── 1. Patch db_postprocess.py to accept 4 or 6 element shapes ──
db_path = os.path.join(PADDLEOCR_DIR, 'ppocr/postprocess/db_postprocess.py')
with open(db_path, 'r') as f: db_code = f.read()
_old = 'src_h, src_w, ratio_h, ratio_w = shape_list[batch_index]'
_new = 'src_h, src_w, ratio_h, ratio_w = shape_list[batch_index][:4]'
if _old in db_code:
    db_code = db_code.replace(_old, _new)
    with open(db_path, 'w') as f: f.write(db_code)
    print('✓ Patched db_postprocess.py (shape[:4])')
else:
    print('✓ db_postprocess.py already patched or not needed')

# ── 2. Patch eval_det_iou.py for Shapely 2.x (Python 3.12) ──
import subprocess as _sp
_sp.run(['git', 'checkout', 'HEAD', '--', 'ppocr/metrics/eval_det_iou.py'],
        cwd=PADDLEOCR_DIR, capture_output=True)
iou_path = os.path.join(PADDLEOCR_DIR, 'ppocr/metrics/eval_det_iou.py')
with open(iou_path, 'r') as f: _lines = f.readlines()
_out = []; _fixed = 0
for _l in _lines:
    if 'Polygon(points).is_valid' in _l:
        _s = ' ' * (len(_l) - len(_l.lstrip()))
        _out.append(f'{_s}if hasattr(points, "reshape") and points.ndim == 1:\n')
        _out.append(f'{_s}    points = points.reshape(-1, 2)\n')
        _out.append(f'{_s}try:\n')
        _out.append(f'{_s}    _poly_valid = Polygon(points).is_valid\n')
        _out.append(f'{_s}except Exception:\n')
        _out.append(f'{_s}    _poly_valid = False\n')
        _out.append(_l.replace('Polygon(points).is_valid', '_poly_valid'))
        _fixed += 1
    else:
        _out.append(_l)
with open(iou_path, 'w') as f: f.writelines(_out)
print(f'✓ eval_det_iou.py patched ({_fixed} sites, with try/except for GEOSException)')

# ── 3. Patch eval_det_iou.py: normalize gt at start of evaluate_image ──
with open(iou_path, 'r') as f: _lines = f.readlines()
_NORM = [
    '        import numpy as _np\n',
    '        _gt_norm = []\n',
    '        for _e in gt:\n',
    '            _pts = _np.asarray(_e[\'points\'])\n',
    '            _ign = _np.asarray(_e[\'ignore\'])\n',
    '            if _pts.ndim == 3:  # (N,4,2) — unwrap per polygon\n',
    '                for _i in range(len(_pts)):\n',
    '                    _gt_norm.append({\'points\': _pts[_i], \'text\': _e.get(\'text\',\'\'), \'ignore\': bool(_ign.flat[_i])})\n',
    '            else:\n',
    '                _ign_s = bool(_ign.flat[0]) if _ign.ndim > 0 else bool(_ign)\n',
    '                _gt_norm.append({\'points\': _pts, \'text\': _e.get(\'text\',\'\'), \'ignore\': _ign_s})\n',
    '        gt = _gt_norm\n',
]
_out2 = []; _f2 = 0
for _l in _lines:
    _out2.append(_l)
    if 'def evaluate_image(self, gt, pred):' in _l:
        _out2.extend(_NORM); _f2 += 1
with open(iou_path, 'w') as f: f.writelines(_out2)
print(f'✓ eval_det_iou.py gt-normalize patch ({_f2} site)')

# ── 4. Ensure config has correct eval settings ──
with open(CONFIG, 'r') as f: cfg = yaml.safe_load(f)
cfg['Global']['eval_batch_step'] = [0, 50]
cfg['Global']['cal_metric_during_train'] = False
cfg['Global']['checkpoints'] = None
for t in cfg['Eval']['dataset']['transforms']:
    if 'DetResizeForTest' in t:
        t['DetResizeForTest'] = None
        break
with open(CONFIG, 'w') as f: yaml.safe_dump(cfg, f, sort_keys=False)
print('✓ Config verified (eval_batch_step=50, cal_metric_during_train=False)')

# ── 5. Patch det_basic_loss.py ──
import re as _re
_bl_path = os.path.join(PADDLEOCR_DIR, 'ppocr/losses/det_basic_loss.py')
with open(_bl_path, 'r') as f: _bl = f.read()

# ── (A) Patch BalanceLoss.forward ──
_FIXED_FORWARD = '''    def forward(self, pred, gt, mask=None):
        """
        The BalanceLoss for Differentiable Binarization text detection
        args:
            pred (variable): predicted feature maps.
            gt (variable): ground truth feature maps.
            mask (variable): masked maps.
        return: (variable) balanced loss
        """
        import numpy as _np
        _EPS = 1e-6

        # Clean gt: NaN/inf → 0.0, then clip to [0, 1]
        _bad_gt = paddle.logical_or(paddle.isnan(gt), paddle.isinf(gt))
        gt_c = paddle.where(_bad_gt, paddle.zeros_like(gt), gt)
        gt_c = paddle.clip(gt_c, 0.0, 1.0)

        # Clean mask: NaN/inf → 0.0, then clip to [0, 1]
        _bad_mask = paddle.logical_or(paddle.isnan(mask), paddle.isinf(mask))
        mask_c = paddle.where(_bad_mask, paddle.zeros_like(mask), mask)
        mask_c = paddle.clip(mask_c, 0.0, 1.0)

        # Clean pred: NaN/inf → 0.5, then clip to [eps, 1-eps]
        _bad_pred = paddle.logical_or(paddle.isnan(pred), paddle.isinf(pred))
        pred_c = paddle.where(_bad_pred, paddle.full_like(pred, 0.5), pred)
        pred_c = paddle.clip(pred_c, _EPS, 1.0 - _EPS)

        # Compute balance counts in numpy (no gradient needed for counts)
        _gt_np   = gt_c.numpy().astype(_np.float64)
        _mask_np = mask_c.numpy().astype(_np.float64)
        _gt_w    = _gt_np * _mask_np
        _neg_w   = (1.0 - _gt_np) * _mask_np
        _max_n   = int(_gt_w.size)
        positive_count = min(int(_gt_w.sum()),  _max_n)
        negative_count = min(int(_neg_w.sum()), int(positive_count * self.negative_ratio), _max_n)

        # Positive/negative weight maps (gradient-connected paddle tensors)
        positive = gt_c * mask_c
        negative = (1.0 - gt_c) * mask_c

        # Compute loss — gradient flows through pred_c → pred
        loss = self.loss(pred_c, gt_c, mask=mask_c)

        if not self.balance_loss:
            return loss

        positive_loss = positive * loss
        negative_loss = negative * loss
        negative_loss = paddle.reshape(negative_loss, shape=[-1])
        if negative_count > 0:
            sort_loss = negative_loss.sort(descending=True)
            negative_loss = sort_loss[:negative_count]
            balance_loss = (positive_loss.sum() + negative_loss.sum()) / (
                positive_count + negative_count + self.eps
            )
        else:
            balance_loss = positive_loss.sum() / (positive_count + self.eps)
        if self.return_origin:
            return balance_loss, loss

        return balance_loss

'''
_pattern = r'    def forward\(self, pred, gt, mask=None\):.*?(?=\nclass |\ndef )'
if _re.search(_pattern, _bl, _re.DOTALL):
    _bl = _re.sub(_pattern, _FIXED_FORWARD, _bl, flags=_re.DOTALL)
    print('✓ BalanceLoss.forward rewritten (paddle.where/isnan — gradient-safe)')
else:
    print('⚠️  BalanceLoss pattern not matched — see snippet:')
    print(_bl[_bl.find('class BalanceLoss'):_bl.find('class BalanceLoss')+300])

# ── (B) BCELoss: NaN guard before F.binary_cross_entropy ──
_bce_old = 'loss = F.binary_cross_entropy(input, label, reduction=self.reduction)'
_bce_new = (
    '_bad_in = paddle.logical_or(paddle.isnan(input), paddle.isinf(input))\n'
    '        input = paddle.where(_bad_in, paddle.full_like(input, 0.5), input)\n'
    '        input = paddle.clip(input, min=1e-6, max=1.0 - 1e-6)\n'
    '        loss = F.binary_cross_entropy(input, label, reduction=self.reduction)'
)
if _bce_old in _bl:
    _bl = _bl.replace(_bce_old, _bce_new)
    print('✓ BCELoss.forward: paddle.where + clip guard added')
else:
    print('✓ BCELoss already patched or not found')

# ── (C) DiceLoss: assert loss <= 1 → paddle.clip ──
if 'assert loss <= 1' in _bl:
    _bl = _bl.replace('assert loss <= 1', 'loss = paddle.clip(loss, max=1.0)')
    print('✓ DiceLoss assert → paddle.clip')
else:
    print('✓ DiceLoss already patched')

# ── (D) MaskL1Loss: NaN guard before L1 computation ──
# The actual signature uses self.eps and calls paddle.mean(loss).
# Replace the entire forward body.
_l1_old = ('def forward(self, pred, gt, mask):\n'
           '        """\n'
           '        Mask L1 Loss\n'
           '        """\n'
           '        loss = (paddle.abs(pred - gt) * mask).sum() / (mask.sum() + self.eps)\n'
           '        loss = paddle.mean(loss)\n'
           '        return loss')
_l1_new = ('def forward(self, pred, gt, mask):\n'
           '        """\n'
           '        Mask L1 Loss\n'
           '        """\n'
           '        _bad_p = paddle.logical_or(paddle.isnan(pred), paddle.isinf(pred))\n'
           '        pred = paddle.where(_bad_p, paddle.zeros_like(pred), pred)\n'
           '        _bad_g = paddle.logical_or(paddle.isnan(gt), paddle.isinf(gt))\n'
           '        gt = paddle.where(_bad_g, paddle.zeros_like(gt), gt)\n'
           '        _bad_m = paddle.logical_or(paddle.isnan(mask), paddle.isinf(mask))\n'
           '        mask = paddle.where(_bad_m, paddle.zeros_like(mask), mask)\n'
           '        loss = (paddle.abs(pred - gt) * mask).sum() / (mask.sum() + self.eps)\n'
           '        loss = paddle.mean(loss)\n'
           '        return loss')
if _l1_old in _bl:
    _bl = _bl.replace(_l1_old, _l1_new)
    print('✓ MaskL1Loss.forward: NaN guard added (fixes loss_threshold_maps)')
else:
    print('⚠️  MaskL1Loss pattern not matched — printing context:')
    _idx = _bl.find('class MaskL1Loss')
    print(_bl[_idx:_idx+500] if _idx >= 0 else '(class not found)')

with open(_bl_path, 'w') as f: f.write(_bl)

# ── 6. Patch det_db_loss.py: sanitise pred maps before all loss calls ──
_db_path = os.path.join(PADDLEOCR_DIR, 'ppocr/losses/det_db_loss.py')
with open(_db_path, 'r') as f: _db = f.read()

_DB_GUARD = (
    '        # ── Sanitise pred maps before any loss computation ──\n'
    '        _maps = predicts[\'maps\']\n'
    '        _bad  = paddle.logical_or(paddle.isnan(_maps), paddle.isinf(_maps))\n'
    '        predicts[\'maps\'] = paddle.where(_bad, paddle.full_like(_maps, 0.5), _maps)\n'
)
_db_trigger = 'def forward(self, predicts, labels):'
if _db_trigger in _db and '# ── Sanitise pred maps' not in _db:
    _db = _db.replace(_db_trigger + '\n', _db_trigger + '\n' + _DB_GUARD)
    print('✓ DBLoss.forward: pred maps NaN guard added (fixes loss_binary_maps)')
else:
    print('✓ DBLoss already patched or pattern not found')

with open(_db_path, 'w') as f: f.write(_db)

# ── 7. Always start fresh from pretrained model ──
# Any checkpoint in det_finetune/ was from a broken NaN run — wiped in step 0.
print('🆕 Starting fresh from pretrained model')
!python tools/train.py -c {CONFIG}


## Step 9 — Export Trained Model


In [ ]:
WORK_DIR      = '/content/workspace'
PADDLEOCR_DIR = '/content/PaddleOCR'
CONFIG        = os.path.join(WORK_DIR, 'det_icdar_finetune.yml')
BEST_MODEL    = os.path.join(WORK_DIR, 'output/det_finetune/best_accuracy')
INFER_OUT     = os.path.join(WORK_DIR, 'output/det_inference_finetuned')

import os
os.chdir(PADDLEOCR_DIR)

!python tools/export_model.py \
    -c {CONFIG} \
    -o Global.pretrained_model={BEST_MODEL} \
       Global.save_inference_dir={INFER_OUT}

print('Export complete!')


## Step 10 — Save Everything to Drive ✅
Saves checkpoints and the final inference model back to your Google Drive.


In [ ]:
import shutil, os
DRIVE_ROOT = '/content/drive/MyDrive/Rumsey_OCR'
WORK_DIR   = '/content/workspace'

DRIVE_OUT = os.path.join(DRIVE_ROOT, 'output')
os.makedirs(DRIVE_OUT, exist_ok=True)

# Save training checkpoints
src_ckpt = os.path.join(WORK_DIR, 'output/det_finetune')
dst_ckpt = os.path.join(DRIVE_OUT, 'det_finetune')
if os.path.exists(src_ckpt):
    shutil.copytree(src_ckpt, dst_ckpt, dirs_exist_ok=True)
    print(f'✅ Checkpoints saved to Drive')

# Save inference model
src_inf = os.path.join(WORK_DIR, 'output/det_inference_finetuned')
dst_inf = os.path.join(DRIVE_OUT, 'det_inference_finetuned')
if os.path.exists(src_inf):
    shutil.copytree(src_inf, dst_inf, dirs_exist_ok=True)
    print(f'✅ Inference model saved to Drive')

print(f'\nAll outputs at: {DRIVE_OUT}')
print('Download det_inference_finetuned/ to your laptop when ready.')


## ✅ All Done!

After the notebook completes:
1. Download `output/det_inference_finetuned/` from Drive to your laptop
2. Place it in `Rumsey_Map_OCR/output/det_inference_finetuned/`
3. Run `step4_evaluate_and_infer.py` on your laptop to see the improved results

**Expected improvement: 27% → 50-60%+ recall** 🎯
